# Convert song_info_map to fret + strings

Reads `song_info_map.json` and converts each event's `pitches + strings` to ukulele `fret4` using the same logic as `pykt/preprocess/yousician_preprocess.py`.

Output: `song_info_fret_strings_map.json`.

In [9]:
from __future__ import annotations

import json
from pathlib import Path

OPEN_MIDI_BY_STRING = {0: 69, 1: 64, 2: 60, 3: 67}

IN_PATH = Path("song_info_map.json")
OUT_PATH = Path("song_info_fret_strings_map.json")

In [10]:
def _to_int_list(x):
    """Match preprocess parser: list/int/str with [] and comma/^ separators."""
    if isinstance(x, list):
        return [int(v) for v in x]
    s = str(x).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]
    s = s.replace(",", "^")
    if not s:
        return []
    return [int(v) for v in s.split("^") if v != ""]


def _cumulative_pitch_list(x):
    vals = _to_int_list(x)
    if not vals:
        return []
    out = [vals[0]]
    for d in vals[1:]:
        out.append(out[-1] + d)
    return out


def _fret4_from_pitch_and_string(pitch_x, string_x):
    mids = _cumulative_pitch_list(pitch_x)
    strs = _to_int_list(string_x)
    if len(mids) != len(strs):
        raise ValueError(f"length mismatch: mids={mids!r}, strings={strs!r}")
    fret4 = [0, 0, 0, 0]
    for m, s in zip(mids, strs):
        if s not in OPEN_MIDI_BY_STRING:
            raise ValueError(f"illegal string index {s}, expected 0..3")
        fret4[s] = int(m) - OPEN_MIDI_BY_STRING[s]
    return fret4


def _canon_strings(string_x):
    # sort ascending per event, e.g. [1,3,2] -> [1,2,3]
    return sorted(_to_int_list(string_x))

In [11]:
with IN_PATH.open("r", encoding="utf-8") as f:
    src = json.load(f)

dst = {}
bad = []

for key, item in src.items():
    pitches_seq = item.get("pitches") or []
    strings_seq = item.get("strings") or []
    duration_seq = item.get("duration") or []
    diff = item.get("difficulty_level")

    n = min(len(pitches_seq), len(strings_seq), len(duration_seq))
    frets = []
    strings_out = []

    ok = True
    for i in range(n):
        p = pitches_seq[i]
        s = strings_seq[i]
        try:
            fret4 = _fret4_from_pitch_and_string(p, s)
            s_list = _canon_strings(s)
        except Exception as e:
            bad.append({"key": key, "index": i, "error": str(e)})
            ok = False
            break

        frets.append(fret4)
        strings_out.append(s_list)

    if not ok:
        continue

    dst[key] = {
        "difficulty_level": diff,
        "fret": frets,
        "strings": strings_out,
        "duration": duration_seq[:n],
    }

with OUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(dst, f, ensure_ascii=False, indent=2)

print("input keys:", len(src))
print("output keys:", len(dst))
print("skipped events:", len(bad))
if bad:
    print("first 3 errors:")
    for x in bad[:3]:
        print(x)

input keys: 973
output keys: 973
skipped events: 0


In [12]:
sample_items = list(dst.items())[:2]
for k, v in sample_items:
    print(k)
    print("  fret[0]:", v["fret"][0] if v["fret"] else None)
    print("  strings[0]:", v["strings"][0] if v["strings"] else None)


sd1DR__eEz_Ci
  fret[0]: [0, 0, 0, 0]
  strings[0]: [0, 1, 2, 3]
  note_count[0]: 4
sz6O_i_eS84w4
  fret[0]: [0, 1, 0, 0]
  strings[0]: [1]
  note_count[0]: 1
